# Training a custom openWakeWord model for "Nova"

Run this in a **Colab GPU runtime** (Runtime -> Change runtime type -> T4 GPU). Official openWakeWord
guidance: this automated training pipeline is Linux-only, which is exactly what a Colab runtime is.

What this notebook does, in order:
1. Install Python 3.11 explicitly (Colab's default Python is too new for one required package).
2. Install training tools (Piper TTS sample generator + openWakeWord's own `train.py`).
3. Download the shared datasets (room impulse responses, background noise, and a large precomputed
   negative-feature set) that openWakeWord's own training recipe uses.
4. Generate synthetic "Nova" voice clips (positive examples) and near-miss/adversarial clips (negative
   examples), augment them with real-world noise/reverb, and extract features.
5. Train a small classifier on those features and export it to `nova.onnx`.
6. Download `nova.onnx` to your machine so it can be wired into `WakeDetector`.

We deliberately **skip the TFLite conversion step** the official notebook also offers - this project's
`openwakeword_wake.py` only ever uses `inference_framework="onnx"`, so there's no reason to install
TensorFlow/`onnx_tf` (which are also the oldest, most fragile pins in the whole pipeline) at all.

**Important about Python versions in this notebook:** `piper-phonemize` only ships wheels for Python
3.9-3.11 (Linux x86_64) - no 3.12+ wheel exists. If Colab's default Python is newer than that, we
install Python 3.11 explicitly and run every shell command through it via `$PY`, rather than fighting
Colab's default interpreter. Notebook *code* cells (the ones with no `!` prefix) still run in Colab's
own default kernel, which is a **different interpreter** from `$PY` - any code cell that needs a
package installs it separately via a plain `!pip install` (no `$PY`) at its own top.

**If you ever have to fully disconnect+delete the runtime** (e.g. to recover from a broken package
install), you MUST re-run every cell from the top again, in order - nothing installed into either
Python (the default kernel's or `python3.11`'s) survives that reset.

## 1. Set up Python 3.11 for every shell command below

In [13]:
!python --version
!sudo apt-get -qq update && sudo apt-get -qq install -y python3.11 python3.11-venv python3.11-dev > /dev/null
!python3.11 -m venv /content/venv311
%env PY=/content/venv311/bin/python
!$PY -m pip install -q -U pip
!echo "Using: $($PY --version)"
!$PY -m pip --version

Python 3.13.16
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
env: PY=/content/venv311/bin/python
Using: Python 3.11.17
pip 26.2.2 from /content/venv311/lib/python3.11/site-packages/pip (python 3.11)


## 2. Install the Piper sample generator (synthesizes the positive "Nova" clips)

In [30]:
import os
if not os.path.exists("piper-sample-generator"):
    !git clone https://github.com/rhasspy/piper-sample-generator
!mkdir -p piper-sample-generator/models
!wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt \
    https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt
!$PY -m pip install -q piper-phonemize webrtcvad piper-tts

In [35]:
%%writefile piper-sample-generator/generate_samples.py
# Shim for openwakeword's train.py, which still does `from generate_samples import
# generate_samples` expecting the OLD flat-file layout of this repo. The repo has since been
# restructured into a proper `piper_sample_generator` package with a changed function signature
# (now requires an explicit `model=` path, uses `slerp_weights` instead of some old params, and
# doesn't accept `auto_reduce_batch_size` at all) - this wraps the new function to match what
# train.py actually calls.
#
# It also resamples every generated clip to 16kHz afterward: the Piper voice synthesizes at its
# own native sample rate (not necessarily 16kHz), and the new generate_samples has no resampling
# built in, but openWakeWord's augment_clips strictly requires 16kHz audio throughout.
from math import gcd
from pathlib import Path

import numpy as np
import soundfile as sf
from scipy.signal import resample_poly
from piper_sample_generator.__main__ import generate_samples as _real_generate_samples

_MODEL_PATH = Path(__file__).resolve().parent / "models" / "en_US-libritts_r-medium.pt"
_TARGET_SR = 16000


def _resample_dir_to_16k(output_dir):
    for wav_path in Path(output_dir).glob("*.wav"):
        data, sr = sf.read(wav_path)
        if sr == _TARGET_SR:
            continue
        g = gcd(sr, _TARGET_SR)
        data_16k = resample_poly(data, _TARGET_SR // g, sr // g)
        sf.write(wav_path, data_16k.astype(np.float32), _TARGET_SR)


def generate_samples(text, max_samples=None, batch_size=1, noise_scales=(0.667,),
                      noise_scale_ws=(0.8,), length_scales=(0.75, 1, 1.25),
                      output_dir=".", file_names=None, **kwargs):
    kwargs.pop("auto_reduce_batch_size", None)  # not supported by the new API; harmless to drop
    result = _real_generate_samples(
        text=text, output_dir=output_dir, model=str(_MODEL_PATH),
        max_samples=max_samples, file_names=file_names, batch_size=batch_size,
        length_scales=tuple(length_scales), noise_scales=tuple(noise_scales),
        noise_scale_ws=tuple(noise_scale_ws),
    )
    _resample_dir_to_16k(output_dir)
    return result

Overwriting piper-sample-generator/generate_samples.py


## 3. Install openWakeWord's own training code and the rest of the training-only dependencies

`python3.11` is a fresh interpreter with nothing installed yet - not even `torch` - since Colab's
preinstalled packages belong to its *default* Python, not this one. Install everything explicitly.
Pinned versions below match openWakeWord's own official training notebook - these are old on purpose
(the training pipeline itself hasn't needed newer versions), and none of this ever touches the shipped
app's own `.venv` since this all runs inside Colab's own disposable environment.

`soundfile` is included here too - it's used later by the AudioSet-extraction script (Step 4), which
also runs through `$PY`.

In [28]:
!$PY -m pip install -q "numpy<2" "pandas<2.2" "scipy<1.15"
!$PY -m pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!$PY -m pip install -q openwakeword
!$PY -m pip install -q mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 speechbrain==0.5.14 \
    audiomentations==0.33.0 torch-audiomentations==0.11.0 acoustics==0.2.6 pronouncing==0.2.0 \
    datasets==2.14.6 "pyarrow<14" deep-phonemizer==0.0.19 huggingface_hub pyyaml "scipy<1.15" soundfile

# Re-assert the fragile pins LAST, with --force-reinstall. A plain `pip install torch torchaudio`
# with no --upgrade flag is a NO-OP if any version is already installed, regardless of whether it's
# the right matching version - which is exactly what let a mismatched torchaudio (pulled in
# transitively by speechbrain/torch-audiomentations above) silently survive past this point before.
# --force-reinstall guarantees a real reinstall of a matched pair from the same index this time.
# scipy<1.15 is needed because `acoustics==0.2.6` (ancient, ~2017) imports scipy.special.sph_harm,
# which SciPy removed in 1.15 (renamed to sph_harm_y) - without the pin, pip grabs the latest scipy.
!$PY -m pip install -q "numpy<2" "pandas<2.2" "scipy<1.15"
!$PY -m pip install -q --force-reinstall --no-deps torch torchaudio --index-url https://download.pytorch.org/whl/cu121

# Downloads openWakeWord's shared melspectrogram + embedding models (used both for training-time
# feature extraction and at inference time by WakeDetector - the same frozen models either way).
!$PY -c "from openwakeword.utils import download_models; download_models()"

## 4. Download the shared datasets (room impulse responses, background noise, precomputed negative features)

These are the real multi-GB downloads - expect this to take a while depending on Colab's network speed
that session. All sources are openWakeWord's own official, documented ones.

This first cell is a notebook **code cell**, so it runs in Colab's own default kernel - a different
interpreter from `$PY` - which is why it installs its own small, separate set of packages at the top.

In [16]:
!pip install -q huggingface_hub

import os
os.makedirs("mit_rirs", exist_ok=True)

from huggingface_hub import hf_hub_download, list_repo_files
from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()

# --- Room impulse responses (adds realistic room reverb to synthetic clips) - plain .wav files,
# no `datasets` library needed here, which avoids any pyarrow ABI conflict with Colab's preinstalled
# numpy/scipy/pyarrow versions.
rir_files = [f for f in list_repo_files("davidscripka/MIT_environmental_impulse_responses", repo_type="dataset")
             if f.endswith(".wav")]
print(f"downloading {len(rir_files)} RIR files...")
for f in rir_files:
    hf_hub_download(repo_id="davidscripka/MIT_environmental_impulse_responses", repo_type="dataset",
                    filename=f, local_dir="mit_rirs")
print("RIR download done.")

# --- Precomputed negative-audio features (generic "random speech/noise" negatives) and the
#     false-positive validation set used only to pick the best checkpoint, not for training itself.
train_features_path = hf_hub_download(
    repo_id="davidscripka/openwakeword_features", repo_type="dataset",
    filename="openwakeword_features_ACAV100M_2000_hrs_16bit.npy")
val_features_path = hf_hub_download(
    repo_id="davidscripka/openwakeword_features", repo_type="dataset",
    filename="validation_set_features.npy")
print("train features:", train_features_path)
print("val features:", val_features_path)

downloading 270 RIR files...
RIR download done.
train features: /root/.cache/huggingface/hub/datasets--davidscripka--openwakeword_features/snapshots/985bf1b47e7f19c07741af82bfe32d5a9dc56096/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
val features: /root/.cache/huggingface/hub/datasets--davidscripka--openwakeword_features/snapshots/985bf1b47e7f19c07741af82bfe32d5a9dc56096/validation_set_features.npy


**Background noise.** The official recipe uses real AudioSet clips here, but that dataset's storage
format has changed twice during this notebook's development (flat `.tar` -> Parquet shards) and its
embedded metadata is now written in a schema our pinned, old `datasets==2.14.6` can't parse at all -
a real compatibility dead end, not something a version pin fixes. Given the RIRs (real room reverb)
and the official 2000-hour precomputed ACAV100M negative-feature set already provide substantial
negative-audio diversity, this cell generates simple synthetic noise instead - zero external
dependencies, so it can't break the same way. `background_paths` only needs *some* valid 16kHz audio
in it; it doesn't have to be AudioSet specifically.

In [17]:
%%writefile generate_background_noise.py
import os
import numpy as np
import soundfile as sf

os.makedirs("audioset_16k", exist_ok=True)
rng = np.random.default_rng(0)
SR = 16000
N_CLIPS = 300

for i in range(N_CLIPS):
    dur_s = rng.uniform(3, 8)
    n = int(dur_s * SR)
    white = rng.normal(0, 1, n)
    # cheap pink-ish noise: cumulative sum of white noise, then normalize - gives a more
    # "room tone" / "distant traffic" character than pure white noise, without needing any
    # external library or dataset.
    pink = np.cumsum(white)
    pink = pink / (np.max(np.abs(pink)) + 1e-9) * 0.3
    sf.write(f"audioset_16k/noise_{i}.wav", pink.astype(np.float32), SR)

print(f"wrote {N_CLIPS} synthetic background noise clips")

Writing generate_background_noise.py


In [18]:
!$PY generate_background_noise.py

wrote 300 synthetic background noise clips


## 5. The training config for "Nova"

`target_phrase` is the bare word "nova" (matching how `hey_jarvis`/`hey_mycroft` are themselves really
trained on the core name, with "hey"/"hi" handled separately by the project's own text-based check).
`custom_negative_phrases` is deliberately loaded with real words that contain or sound like "nova" -
**"innovate", "innovation", "renovate", "renovation", "novation"** all have "nova" embedded inside them
phonetically, and "Nova Scotia", "nouveau", "over", "sofa" are close enough to be worth biasing against -
so the model is explicitly taught these are NOT the wake word, not left to guess.

This is also a notebook **code cell** (Colab's default kernel) - PyYAML ships preinstalled there, so
no extra install is needed for this one. It depends on `train_features_path`/`val_features_path` from
Step 4 above, so make sure that cell finished (printed both `features:` lines) before running this one.

In [41]:
import yaml

config = {
    "piper_sample_generator_path": "./piper-sample-generator",
    "output_dir": "./my_custom_model",
    "model_name": "nova",
    "target_phrase": ["nova"],
    "custom_negative_phrases": [
        "innovate", "innovation", "renovate", "renovation", "novation",
        "nova scotia", "nouveau", "over", "sofa", "no", "over there",
    ],
    "n_samples": 2000,
    "n_samples_val": 500,
    "tts_batch_size": 50,
    "augmentation_rounds": 1,
    "augmentation_batch_size": 32,
    "rir_paths": ["./mit_rirs/16khz"],
    "background_paths": ["./audioset_16k"],
    "background_paths_duplication_rate": [1],
    "false_positive_validation_data_path": val_features_path,
    "feature_data_files": {
        "ACAV100M": train_features_path,
    },
    "batch_n_per_class": {
        "positive": 200, "adversarial_negative": 200, "ACAV100M": 200,
    },
    "model_type": "dnn",
    "layer_size": 32,
    "steps": 10000,
    "max_negative_weight": 1500,
    "target_false_positives_per_hour": 0.2,
}

with open("nova.yml", "w") as f:
    yaml.dump(config, f)

print(open("nova.yml").read())

augmentation_batch_size: 32
augmentation_rounds: 1
background_paths:
- ./audioset_16k
background_paths_duplication_rate:
- 1
batch_n_per_class:
  ACAV100M: 200
  adversarial_negative: 200
  positive: 200
custom_negative_phrases:
- innovate
- innovation
- renovate
- renovation
- novation
- nova scotia
- nouveau
- over
- sofa
- 'no'
- over there
false_positive_validation_data_path: /root/.cache/huggingface/hub/datasets--davidscripka--openwakeword_features/snapshots/985bf1b47e7f19c07741af82bfe32d5a9dc56096/validation_set_features.npy
feature_data_files:
  ACAV100M: /root/.cache/huggingface/hub/datasets--davidscripka--openwakeword_features/snapshots/985bf1b47e7f19c07741af82bfe32d5a9dc56096/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
layer_size: 32
max_negative_weight: 1500
model_name: nova
model_type: dnn
n_samples: 2000
n_samples_val: 500
output_dir: ./my_custom_model
piper_sample_generator_path: ./piper-sample-generator
rir_paths:
- ./mit_rirs/16khz
steps: 10000
target_false_positi

## 6. Run the three training stages

Each stage is resumable/idempotent (it skips work that's already done if you re-run a cell), so if a
stage fails partway (a download hiccup, a Colab disconnect), just re-run that same cell.

In [36]:
# Stage 1: synthesize positive ("Nova") and adversarial-negative voice clips with Piper TTS
# MPLBACKEND=Agg overrides Colab's own inline-notebook matplotlib backend (inherited from the shell
# environment), which torchmetrics' plotting utilities can't use outside a real notebook kernel.
!MPLBACKEND=Agg $PY -m openwakeword.train --training_config nova.yml --generate_clips

/content/venv311/lib/python3.11/site-packages/torch_audiomentations/utils/io.py:27: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("soundfile")
torchvision is not available - cannot save figures
INFO:root:##################################################
Generating positive clips for training
##################################################
DEBUG:piper_sample_generator.__main__:Loading /content/piper-sample-generator/models/en_US-libritts_r-medium.pt
INFO:piper_sample_generator.__main__:Successfully loaded the model
DEBUG:piper_sample_generator.__main__:CUDA available, using GPU
DEBUG:piper_sample_generator.__main__:Batch 1/40 complete
DEBUG:piper_sample_generator.__main__:Batch 2/40 complete
DEBUG:piper_sample_generator.__main__:Batch 3/40 complete
DEBUG:piper_sample_generator.__main__:Batch 4/40 complete
DEBUG:piper_sample_generator.__main__:Ba

In [42]:
# Stage 2: add real reverb/background noise, extract embedding features, save to .npy
# --overwrite forces regeneration: the previous run's crash (wrong sample rate) left a
# positive_features_train.npy partially written, which the stage's own skip-if-exists check
# would otherwise treat as "already done" and silently skip past.
!MPLBACKEND=Agg $PY -m openwakeword.train --training_config nova.yml --augment_clips --overwrite

/content/venv311/lib/python3.11/site-packages/torch_audiomentations/utils/io.py:27: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("soundfile")
torchvision is not available - cannot save figures
INFO:root:##################################################
Computing openwakeword features for generated samples
##################################################
/content/venv311/lib/python3.11/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:197: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
/content/venv311/lib/python3.11/site-packages/torch_audiomentations/core/transforms_interface.py:77: FutureWarning: Transforms now expect an `output_type` argument that currently defaults to 'tensor', will default to 'di

In [47]:
!$PY -m pip install -q onnx
!$PY -m pip install -q "numpy<2" "pandas<2.2" "scipy<1.15"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ml-dtypes 0.6.0 requires numpy>=2.0.0, but you have numpy 1.26.4 which is incompatible.


In [48]:
# Stage 3: train the small classifier (GPU-accelerated) and export to ONNX
!MPLBACKEND=Agg $PY -m openwakeword.train --training_config nova.yml --train_model

/content/venv311/lib/python3.11/site-packages/torch_audiomentations/utils/io.py:27: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("soundfile")
torchvision is not available - cannot save figures
INFO:root:##################################################
Starting training sequence 1...
##################################################

INFO:root:##################################################
Starting training sequence 2...
##################################################
INFO:root:Increasing weight on negative examples to reduce false positives...

INFO:root:##################################################
Starting training sequence 3...
##################################################
INFO:root:Increasing weight on negative examples to reduce false positives...

INFO:root:Merging checkpoints above the 90th percentile into single model..

## 7. Download the trained model

`my_custom_model/nova.onnx` is the only file we actually need back on your machine. Note the training
log's own reported validation recall and false-positive-per-hour numbers from the cell above before
trusting this model - they're printed near the end of Stage 3's output.

In [49]:
from google.colab import files
files.download("my_custom_model/nova.onnx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [52]:
import requests
with open("my_custom_model/nova.onnx", "rb") as f:
    r = requests.post("https://0x0.st", files={"file": f})
print(r.status_code)
print(r.text)

503
uploads disabled because itâ€™s been almost nothing but AI botnet spam for the past few months. will be back with a few changes at some point. no ETA.


In [53]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy("my_custom_model/nova.onnx", "/content/drive/MyDrive/nova.onnx")
print("copied to Google Drive — open drive.google.com in any browser and download nova.onnx from My Drive")

Mounted at /content/drive
copied to Google Drive — open drive.google.com in any browser and download nova.onnx from My Drive
